# Sampling Methods

The notebook follows the stdlib sampling functions in `code/sampling.py` and reports numerical summaries instead of plotting.

In [ ]:
from pathlib import Path
import sys
candidates = (Path.cwd() / 'code', Path.cwd().parents[1] / 'code', Path.cwd() / 'phases/01-math-foundations/16-sampling-methods/code')
code_dir = next(path for path in candidates if (path / 'sampling.py').is_file())
sys.path.insert(0, str(code_dir))
from sampling import sample_exponential_inverse_cdf, truncated_normal_demo, metropolis_hastings, temperature_distribution, top_k_distribution, top_p_distribution, reparam_sample, reparam_gradient
import math
import random


## Build It: transform a uniform draw

For exponential rate `lambda=1`, a fixed `U=0.5` maps to `-ln(0.5)`. Rejection sampling returns only values in the requested truncation interval.

In [ ]:
original_random = random.random
random.random = lambda: 0.5
inverse = sample_exponential_inverse_cdf(1.0)
random.random = original_random
random.seed(4)
truncated, acceptance = truncated_normal_demo(0.0, 1.0, -1.0, 2.0, n=80)
inverse, min(truncated), max(truncated), acceptance


## Use It: MCMC and decoding distributions

Metropolis-Hastings returns exactly `n_samples` states after discarding `burn_in`. Temperature, top-k, and top-p all renormalize probability vectors.

In [ ]:
random.seed(6)
chain, acceptance = metropolis_hastings(lambda x: -0.5 * x * x, 5.0, 100, 20, proposal_std=1.0)
logits = [3.0, 2.0, 1.0, 0.0]
len(chain), acceptance, temperature_distribution(logits, 0.5), top_k_distribution(logits, 2), top_p_distribution(logits, 0.8)


## Ship It: differentiable randomness

The reparameterized sample is `z = mu + sigma * epsilon`; the returned local derivatives are `dz/dmu=1` and `dz/dsigma=epsilon`.

In [ ]:
random.seed(7)
z, epsilon = reparam_sample(2.0, 0.5)
z, epsilon, reparam_gradient(epsilon)


## Exercise

Run the logits fixture with temperatures `0.5` and `2.0`; count nonzero entries for `top_k_distribution(..., 2)` and `top_p_distribution(..., 0.8)`. Explain why the top-p candidate count is data-dependent.